<a href="https://colab.research.google.com/github/rathore-rl1lmg1/Cooding-Playground/blob/main/frozen_lake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import gymnasium as gym
import random

# Create environment (4x4 )
env = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=False)

# State and action space sizes
state_space_size = env.observation_space.n
action_space_size = env.action_space.n

# Initialize Q-table
q_table = np.zeros((state_space_size, action_space_size))

# Hyperparameters
num_episodes = 50000
max_steps_per_episode = 100

learning_rate = 0.001
discount_rate = 0.99

exploration_rate = 1.0
min_exploration_rate = 0.1
max_exploration_rate = 1.0
exploration_decay_rate = 0.00001

rewards_all_episodes = []

# ---------------- Q-learning algorithm ----------------
for episode in range(num_episodes):
    state, _ = env.reset()
    rewards_current_episode = 0

    for step in range(max_steps_per_episode):

        # Exploration vs exploitation
        if random.uniform(0, 1) < exploration_rate:
            action = env.action_space.sample()     # explore
        else:
            action = np.argmax(q_table[state, :])  # exploit

        new_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # Q-table update
        q_table[state, action] = q_table[state, action] * (1 - learning_rate) + \
            learning_rate * (reward + discount_rate * np.max(q_table[new_state, :]))

        state = new_state
        rewards_current_episode += reward

        if done:
            break

    # Exploration rate decay
    exploration_rate = min_exploration_rate + \
        (max_exploration_rate - min_exploration_rate) * np.exp(-exploration_decay_rate * episode)

    rewards_all_episodes.append(rewards_current_episode)

# ---------------- Results ----------------
# Average reward per 1000 episodes
rewards_per_thousand = np.array_split(
    np.array(rewards_all_episodes), num_episodes // 1000
)

print("Average reward per thousand episodes:")
count = 1000
for r in rewards_per_thousand:
    print(f"{count}: {np.mean(r):.3f}")
    count += 1000

# Print learned Q-table
print("\nFinal Q-table:")
print(q_table)

# ---------------- Test trained agent ----------------
state, _ = env.reset()
done = False
total_reward = 0

print("\nTesting trained agent:")
env2 = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=False , render_mode='human')
env2.reset()

while not done:
    action = np.argmax(q_table[state, :])
    state, reward, terminated, truncated, _ = env2.step(action)
    done = terminated or truncated
    total_reward += reward
    env2.render()

print("Final reward:", total_reward)


Average reward per thousand episodes:
1000: 0.002
2000: 0.004
3000: 0.000
4000: 0.004
5000: 0.003
6000: 0.006
7000: 0.004
8000: 0.006
9000: 0.007
10000: 0.009
11000: 0.004
12000: 0.016
13000: 0.010
14000: 0.017
15000: 0.022
16000: 0.026
17000: 0.025
18000: 0.027
19000: 0.028
20000: 0.030
21000: 0.036
22000: 0.040
23000: 0.053
24000: 0.047
25000: 0.045
26000: 0.062
27000: 0.055
28000: 0.065
29000: 0.063
30000: 0.081
31000: 0.098
32000: 0.102
33000: 0.116
34000: 0.094
35000: 0.111
36000: 0.132
37000: 0.113
38000: 0.124
39000: 0.159
40000: 0.134
41000: 0.159
42000: 0.139
43000: 0.154
44000: 0.177
45000: 0.177
46000: 0.192
47000: 0.194
48000: 0.180
49000: 0.194
50000: 0.226

Final Q-table:
[[1.82914093e-01 2.63250471e-01 1.87729410e-01 1.85656880e-01]
 [1.33548857e-01 2.67565598e-01 1.39240813e-01 1.38359906e-01]
 [1.11217219e-01 2.72801836e-01 1.21283569e-01 1.11854128e-01]
 [9.39994726e-02 2.78695734e-01 1.10352389e-01 9.54141698e-02]
 [8.16884325e-02 3.05515366e-01 1.06403584e-01 9.6678